# 03_OneHotEncoder.ipynb

# 1. Introduction

## What is OneHotEncoder?

`OneHotEncoder` converts **categorical (text or discrete)** features into numerical features that machine learning models can understand.

Instead of assigning numbers like 0, 1, 2 (which may imply an order), it creates a new binary column for each category.

---

## Why do we need it?

Most machine learning algorithms work only with **numerical data**.

Example

| Color |
| ----- |
| Red   |
| Blue  |
| Green |

Algorithms cannot directly process these string values.

---

## Problem with Label Encoding

Suppose we encode colors as

| Color | Label |
| ----- | ----: |
| Red   |     0 |
| Blue  |     1 |
| Green |     2 |

The model may incorrectly assume

```text
Green > Blue > Red
```

or

```text
Green - Red = 2
```

even though colors have **no natural order**.

---

## Solution: One-Hot Encoding

Instead of one column, create one column for each category.

| Color | Blue | Green | Red |
| ----- | ---: | ----: | --: |
| Red   |    0 |     0 |   1 |
| Blue  |    1 |     0 |   0 |
| Green |    0 |     1 |   0 |

Exactly one column contains **1**, and the rest contain **0**.

Hence the name **One-Hot Encoding**.

---

## Mathematical Intuition

If a feature has **k categories**, OneHotEncoder creates **k binary features**.

Example

```text
Gender

Male
Female
```

↓

| Male | Female |
| ---: | -----: |
|    1 |      0 |
|    0 |      1 |

---

## Advantages

* Converts categorical data into numerical form.
* Prevents artificial ordering between categories.
* Widely used in ML pipelines.
* Works well with most machine learning algorithms.

---

## Disadvantages

* Increases the number of features.
* High-cardinality features can create many new columns.
* Sparse output can consume memory for very large datasets.

---

## When to Use

✅ Nominal categorical features

* Color
* City
* Country
* Department
* Product Category

---

## When Not to Use

❌ Ordinal features

Examples

* Small < Medium < Large
* Low < Medium < High

Use `OrdinalEncoder` instead.

---

## Dummy Variable Trap

Suppose

| Color | Blue | Green | Red |
| ----- | ---: | ----: | --: |
| Blue  |    1 |     0 |   0 |
| Green |    0 |     1 |   0 |
| Red   |    0 |     0 |   1 |

Since

```text
Blue + Green + Red = 1
```

one column is redundant.

For linear models, we often remove one column using

```python
drop="first"
```

to avoid multicollinearity.

---

## LabelEncoder vs OneHotEncoder

| Feature       | LabelEncoder          | OneHotEncoder                    |
| ------------- | --------------------- | -------------------------------- |
| Output        | Single integer column | Multiple binary columns          |
| Creates Order | Yes                   | No                               |
| Best For      | Target labels         | Input categorical features       |
| Nominal Data  | ❌                     | ✅                                |
| Ordinal Data  | ✅                     | ⚠️ Only if order isn't important |

---

# 2. Import & Constructor

## Import

```python
from sklearn.preprocessing import OneHotEncoder
```

Create an object

```python
encoder = OneHotEncoder()
```

---

## Constructor Syntax

```python
OneHotEncoder(
    *,
    categories="auto",
    drop=None,
    sparse_output=True,
    dtype=np.float64,
    handle_unknown="error",
    min_frequency=None,
    max_categories=None,
    feature_name_combiner="concat"
)
```

---

## Constructor Parameters

| Parameter               | Default      | Description                                       | Common Usage                    |
| ----------------------- | ------------ | ------------------------------------------------- | ------------------------------- |
| `categories`            | `"auto"`     | Automatically detects unique categories.          | Keep `"auto"`                   |
| `drop`                  | `None`       | Drops one category to avoid multicollinearity.    | `"first"` for Linear Regression |
| `sparse_output`         | `True`       | Returns a sparse matrix instead of a dense array. | `False` for easier inspection   |
| `dtype`                 | `np.float64` | Data type of encoded values.                      | Keep default                    |
| `handle_unknown`        | `"error"`    | Controls behavior for unseen categories.          | `"ignore"` for deployment       |
| `min_frequency`         | `None`       | Groups infrequent categories together.            | Large datasets                  |
| `max_categories`        | `None`       | Limits the number of output categories.           | High-cardinality features       |
| `feature_name_combiner` | `"concat"`   | Controls generated feature names.                 | Keep default                    |

---

## Important Parameters

### `categories`

Automatically detects categories.

```python
encoder = OneHotEncoder(categories="auto")
```

Or specify them manually.

```python
encoder = OneHotEncoder(
    categories=[["Blue", "Green", "Red"]]
)
```

Useful when you want a fixed category order.

---

### `drop`

Removes one encoded column.

```python
encoder = OneHotEncoder(drop="first")
```

Example

Instead of

| Blue | Green | Red |
| ---- | ----- | --- |
| 1    | 0     | 0   |

becomes

| Green | Red |
| ----- | --- |
| 0     | 0   |

Useful for Linear Regression and Logistic Regression.

---

### `sparse_output`

Default

```python
True
```

Returns a **Sparse Matrix** (memory efficient).

If

```python
encoder = OneHotEncoder(sparse_output=False)
```

returns a normal NumPy array.

Useful while learning and debugging.

---

### `handle_unknown`

Default

```python
handle_unknown="error"
```

If a new unseen category appears during testing,

```text
Purple
```

an error is raised.

Use

```python
handle_unknown="ignore"
```

to safely ignore unseen categories during deployment.

---

### `min_frequency`

Groups rare categories together.

Useful for columns having many uncommon values.

---

### `max_categories`

Limits the maximum number of generated columns.

Useful for high-cardinality features.

---

## Recommended Settings

### For Learning

```python
encoder = OneHotEncoder(
    sparse_output=False
)
```

---

### For Linear Models

```python
encoder = OneHotEncoder(
    drop="first",
    sparse_output=False
)
```

---

### For Deployment

```python
encoder = OneHotEncoder(
    handle_unknown="ignore"
)
```

---


# 3. Methods & Attributes

## Methods

| Method                    | Purpose                                            | Returns             |
| ------------------------- | -------------------------------------------------- | ------------------- |
| `fit(X)`                  | Learns unique categories from the training data.   | Encoder object      |
| `transform(X)`            | Converts categories into one-hot encoded features. | Encoded data        |
| `fit_transform(X)`        | Learns categories and encodes in one step.         | Encoded data        |
| `inverse_transform(X)`    | Converts encoded data back to original categories. | Original categories |
| `get_feature_names_out()` | Returns names of encoded features.                 | Feature name array  |
| `set_output()`            | Sets output type (NumPy/Pandas).                   | Encoder object      |
| `get_metadata_routing()`  | Returns metadata routing information.              | MetadataRouter      |

---

## Most Used Methods

| Method                    | Usage                                |
| ------------------------- | ------------------------------------ |
| `fit()`                   | Learn categories from training data  |
| `transform()`             | Encode validation, test, or new data |
| `fit_transform()`         | Encode training data                 |
| `inverse_transform()`     | Recover original categories          |
| `get_feature_names_out()` | Get names of generated columns       |

---

## Attributes

After calling `fit()`, the encoder stores the following information.

| Attribute                | Description                                                                          |
| ------------------------ | ------------------------------------------------------------------------------------ |
| `categories_`            | Unique categories for each feature.                                                  |
| `drop_idx_`              | Index of the dropped category (if `drop` is used).                                   |
| `feature_names_in_`      | Original feature names (DataFrame only).                                             |
| `n_features_in_`         | Number of input features.                                                            |
| `infrequent_categories_` | Categories grouped as infrequent (when `min_frequency` or `max_categories` is used). |

---

## Example

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "Color": ["Red", "Blue", "Green", "Blue"]
})

encoder = OneHotEncoder(sparse_output=False)

encoder.fit(df)

print(encoder.categories_)
print(encoder.feature_names_in_)
print(encoder.n_features_in_)
print(encoder.get_feature_names_out())
```

Output

```text
categories_ :
[array(['Blue', 'Green', 'Red'], dtype=object)]

feature_names_in_ :
['Color']

n_features_in_ :
1

get_feature_names_out() :
['Color_Blue' 'Color_Green' 'Color_Red']
```

---

## Most Frequently Used Attributes

| Attribute                | Importance |
| ------------------------ | ---------- |
| `categories_`            | ⭐⭐⭐⭐⭐      |
| `feature_names_in_`      | ⭐⭐⭐⭐       |
| `n_features_in_`         | ⭐⭐⭐        |
| `drop_idx_`              | ⭐⭐⭐        |
| `infrequent_categories_` | ⭐⭐         |

> **Note:** Unlike `StandardScaler` or `MinMaxScaler`, `OneHotEncoder` does **not** learn statistical values (mean, std, min, max). It only learns the unique categories present in each feature and uses them to create new binary columns.


# 4. End-to-End Workflow

## Workflow

```text
Load Dataset
      ↓
Train-Test Split
      ↓
Create OneHotEncoder
      ↓
fit_transform(X_train)
      ↓
transform(X_test)
      ↓
Train Model
      ↓
Predict
```

---

## Complete Example

```python
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    "Color": ["Red", "Blue", "Green", "Blue", "Red"],
    "Brand": ["A", "B", "A", "C", "B"]
})

X_train, X_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

X_train_encoded = encoder.fit_transform(X_train)

X_test_encoded = encoder.transform(X_test)

print(X_train_encoded)
```

---

## Important Notes

* Fit the encoder only on the training data.
* Use `transform()` for validation, test, and new data.
* Use `handle_unknown="ignore"` for deployment.
* OneHotEncoder should only be applied to categorical features.
* Save the fitted encoder along with the model.

---

# 5. Common Errors

| Mistake                                                            | Why It Happens                                                                      |
| ------------------------------------------------------------------ | ----------------------------------------------------------------------------------- |
| Using `LabelEncoder` instead of `OneHotEncoder` for input features | LabelEncoder introduces an artificial order between categories.                     |
| Calling `transform()` before `fit()`                               | The encoder has not learned the categories yet.                                     |
| Using `fit_transform()` on the test set                            | Learns categories from the test data, causing data leakage.                         |
| Not handling unknown categories                                    | An unseen category raises an error unless `handle_unknown="ignore"` is used.        |
| Applying OneHotEncoder to ordinal features                         | Ordinal features have a natural order and are better handled by `OrdinalEncoder`.   |
| Encoding the target variable (`y`)                                 | OneHotEncoder is generally used for input features (`X`), not target labels.        |
| Applying OneHotEncoder to numerical features                       | It unnecessarily increases the number of columns without adding useful information. |

---